# Запуск SIR модели с миграцией

**Цель:** Выполнение симуляции SIR модели с миграцией между городами,
сбор данных о динамике эпидемии и визуализация результатов.

## Инициализация проекта и загрузка пакетов

In [1]:
using DrWatson
@quickactivate "project"
using Agents, DataFrames, Plots
using JLD2

include(srcdir("sir_model.jl"))

total_count (generic function with 1 method)

## Параметры эксперимента

In [2]:
params = Dict(
    :Ns => [1000, 1000, 1000],
    :β_und => [0.5, 0.5, 0.5],
    :β_det => [0.05, 0.05, 0.05],
    :infection_period => 14,
    :detection_time => 7,
    :death_rate => 0.02,
    :reinfection_probability => 0.1,
    :Is => [0, 0, 1],
    :seed => 42,
    :n_steps => 100,
)

Dict{Symbol, Any} with 10 entries:
  :Is                      => [0, 0, 1]
  :seed                    => 42
  :infection_period        => 14
  :n_steps                 => 100
  :β_und                   => [0.5, 0.5, 0.5]
  :Ns                      => [1000, 1000, 1000]
  :detection_time          => 7
  :reinfection_probability => 0.1
  :β_det                   => [0.05, 0.05, 0.05]
  :death_rate              => 0.02

## Инициализация модели

In [3]:
model = initialize_sir(; params...)

StandardABM with 3000 agents of type Person
 agents container: Dict
 space: GraphSpace with 3 positions and 3 edges
 scheduler: fastest
 properties: C, infection_period, β_und, Ns, migration_rates, detection_time, reinfection_probability, β_det, death_rate

## Подготовка массивов для хранения данных

In [4]:
times = Int[]
S_vals = Int[]
I_vals = Int[]
R_vals = Int[]
total_vals = Int[]

Int64[]

## Запуск симуляции вручную

In [5]:
for step = 1:params[:n_steps]
    Agents.step!(model, 1)

    push!(times, step)
    push!(S_vals, susceptible_count(model))
    push!(I_vals, infected_count(model))
    push!(R_vals, recovered_count(model))
    push!(total_vals, total_count(model))
end

## Создаём DataFrame для удобства (опционально)

In [6]:
agent_df =
    DataFrame(time = times, susceptible = S_vals, infected = I_vals, recovered = R_vals)
model_df = DataFrame(time = times, total = total_vals)

Row,time,total
,Int64,Int64
1,1,3000
2,2,3000
3,3,3000
4,4,3000
5,5,3000
6,6,3000
7,7,3000
8,8,3000
9,9,3000


## Визуализация

In [7]:
plot(
    agent_df.time,
    agent_df.susceptible,
    label = "Восприимчивые",
    xlabel = "Дни",
    ylabel = "Количество",
)
plot!(agent_df.time, agent_df.infected, label = "Инфицированные")
plot!(agent_df.time, agent_df.recovered, label = "Выздоровевшие")
plot!(agent_df.time, model_df.total, label = "Всего (включая умерших)", linestyle = :dash)
savefig(plotsdir("sir_basic_dynamics.png"))

Could not create decoration from factory! Running with no decorations.


"/home/mmulitina/work/study/2026-1/backup/2026-1-study-simulation-modeling/labs/lab04/project/plots/sir_basic_dynamics.png"

## Сохранение данных

In [8]:
@save datadir("sir_basic_agent.jld2") agent_df
@save datadir("sir_basic_model.jld2") model_df